In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'data' / '3_train.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'data' / '3_val.parquet')

# from sklearn import set_config
# set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = [
    col for col in (train.select_dtypes(include='number').columns)
    if col not in ['event_id', target_col]]  # 'station_id', 'timestamp'는 미포함
origin_cols = wl_cols + rain_cols + env_cols

assert len(num_cols) == 31, "'num_cols' 개수 이상"  # 종속변수 제외됨
assert (set(train.columns) - set(num_cols)) == {'station_id', 'timestamp', 'event_id', target_col}, "'num_cols' 확인필요"
assert set(origin_cols).issubset(num_cols), "'origin_cols' 확인필요"

In [3]:
# ============================================================
# 2. 종속변수 분리
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [12]:
# ============================================================
# 3. 최종 독립변수 정의(OHE 포함)
# ============================================================

# 기존 독립변수 정의
cols = origin_cols + wl_diff_cols

# 독립변수에 OHE 추가하기
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 안띄움
    sparse_output=True  # 희소행렬로 반환(원핫인코딩을 가볍게 만들어줌, 빅데이터에서 필수)
)

ohe.fit(train[['station_id']])  # skleran 인코더 2차원 행렬 필수

assert set(val['station_id']).issubset(set(train['station_id'])), "val에 변환되지 않은 'station_id' 발생 가능"

train_ohe = ohe.transform(train[['station_id']])
val_ohe = ohe.transform(val[['station_id']])
# 여기까지는 변환, 실제 데이터에 합친건 아님

# 합치기
from scipy.sparse import hstack

X_train = hstack([
    train[cols],
    train_ohe
]).tocsr()

X_val = hstack([
    val[cols],
    val_ohe
]).tocsr()

# 행 개수 검증
assert X_train.shape[0] == len(train), "Train 행 개수 불일치"
assert X_val.shape[0] == len(val), "Val 행 개수 불일치"
# 열 개수 검증
assert X_train.shape[1] == len(cols) + train_ohe.shape[1], "Train 열 개수 불일치"
assert X_val.shape[1] == X_train.shape[1], "Train과 Val 열 개수 불일치"

In [14]:
# ============================================================
# 4. Cat 최적화-1: best iteration 탐색
# ============================================================
# 사유: 모델비교에서 트리개수가 기본값의 최대치인 1000으로 확인, 학습이 부족했을 가능성
# iteration=3000, early_stopping_rounds=100, eval_set=(X_val, y_val) 적용 전
# best_iteration이 999(1000번째)인지 확인 필요

# CatBoost 기본모델: iteration=1000으로 best_iteratoin 확인

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

results = []

# 모델링
model1 = CatBoostRegressor(
    verbose=500,
    random_state=42
)

model1.fit(
    X_train, y_train,
    early_stopping_rounds=100,
    eval_set=(X_val, y_val)
)

pred_val = model1.predict(X_val)
pred_train = model1.predict(X_train)

rmse_val = root_mean_squared_error(y_val, pred_val)
rmse_train = root_mean_squared_error(y_train, pred_train)

gap = rmse_val - rmse_train

results.append({
    'Model': 'Cat_iter_1000',
    'best_iteration': model1.get_best_iteration(),
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': gap
    })

print(results[-1])


Learning rate set to 0.139602
0:	learn: 0.3771988	test: 0.3324216	best: 0.3324216 (0)	total: 59.6ms	remaining: 59.6s
500:	learn: 0.2053794	test: 0.1969128	best: 0.1969128 (500)	total: 23.2s	remaining: 23.1s
999:	learn: 0.1901659	test: 0.1950727	best: 0.1950234 (990)	total: 45.5s	remaining: 0us

bestTest = 0.1950233664
bestIteration = 990

Shrink model to first 991 iterations.
{'Model': 'Cat_iter_1000', 'best_iteration': 990, 'val_RMSE': 0.19502336643352713, 'train_RMSE': 0.19040694727729632, 'Gap': 0.0046164191562308166}


In [ ]:
# ============================================================
# 5. Cat 최적화-2: iteration: 3000으로 상향
# ============================================================
# 사유: 모델비교에서 트리개수가 기본값의 최대치인 1000으로 확인, 학습이 부족했을 가능성
# iteration=3000, early_stopping_rounds=100, eval_set=(X_val, y_val) 적용

# CatBoost, iteration=3000으로 best_iteratoin 확인

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# 모델링
model2 = CatBoostRegressor(
    iterations=3000,
    verbose=500,
    random_state=42
)

model2.fit(
    X_train, y_train,
    early_stopping_rounds=100,
    eval_set=(X_val, y_val)
)

pred_val = model2.predict(X_val)
pred_train = model2.predict(X_train)

rmse_val = root_mean_squared_error(y_val, pred_val)
rmse_train = root_mean_squared_error(y_train, pred_train)

gap = rmse_val - rmse_train

results.append({
    'Model': 'Cat_iter_3000',
    'best_iteration': model2.get_best_iteration(),
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': gap
    })

print(results[-1])

Learning rate set to 0.071425
0:	learn: 0.3895086	test: 0.3444559	best: 0.3444559 (0)	total: 55.9ms	remaining: 2m 47s
500:	learn: 0.2173392	test: 0.1993331	best: 0.1993276 (498)	total: 28.6s	remaining: 2m 22s
1000:	learn: 0.2044676	test: 0.1962161	best: 0.1962138 (997)	total: 58.7s	remaining: 1m 57s
1500:	learn: 0.1959571	test: 0.1950167	best: 0.1950026 (1478)	total: 1m 30s	remaining: 1m 30s
2000:	learn: 0.1892894	test: 0.1942970	best: 0.1942691 (1982)	total: 2m 1s	remaining: 1m
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.194134478
bestIteration = 2337

Shrink model to first 2338 iterations.
{'Model': 'Cat_iter_3000', 'best_iteration': 2337, 'val_RMSE': 0.1941344779529384, 'train_RMSE': 0.18545961081761345, 'Gap': 0.008674867135324948}


In [18]:
# ============================================================
# 6. Cat 최적화-3: OHE 대신 cat모델 자체 인코더 사용
# ============================================================

# 독립변수 정의
cols2 = origin_cols + wl_diff_cols + ['station_id']
X_train = train[cols2].copy()  # 'station_id' 문자열 변환 예정이므로 copy() 필요
X_val = val[cols2].copy()

X_train['station_id'] = X_train['station_id'].astype(str)
X_val['station_id'] = X_val['station_id'].astype(str)

# 모델링
model3 = CatBoostRegressor(
    iterations=3000,
    verbose=500,
    random_state=42
)

model3.fit(
    X_train, y_train,
    cat_features=['station_id'],  # 자체 인코더 설정 필수
    early_stopping_rounds=100,
    eval_set=(X_val, y_val)
)

pred_val = model3.predict(X_val)
pred_train = model3.predict(X_train)

rmse_val = root_mean_squared_error(y_val, pred_val)
rmse_train = root_mean_squared_error(y_train, pred_train)

gap = rmse_val - rmse_train

results.append({
    'Model': 'Cat_no_ohe',
    'best_iteration': model3.get_best_iteration(),
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': gap
    })

print(results[-1])

Learning rate set to 0.071425
0:	learn: 0.3895443	test: 0.3446945	best: 0.3446945 (0)	total: 278ms	remaining: 13m 53s
500:	learn: 0.2155696	test: 0.1961997	best: 0.1961997 (500)	total: 1m 28s	remaining: 7m 20s
1000:	learn: 0.2029600	test: 0.1931596	best: 0.1931596 (1000)	total: 2m 54s	remaining: 5m 48s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1924565473
bestIteration = 1328

Shrink model to first 1329 iterations.
{'Model': 'Cat_no_ohe', 'best_iteration': 1328, 'val_RMSE': 0.1924565473356066, 'train_RMSE': 0.19649307093895113, 'Gap': -0.004036523603344527}


### 최종 모델
**iteration=3000**

**CatBoost 자체 인코더 사용**

In [ ]:
# ============================================================
# 7. Cat 최적화-3: K-Fold 교차검증
# ============================================================

# 각각 예측데이터 저장해서 OOF도 만들고 플롯도 해야함

# ------------------------------------------------------------
# 3-1. 기존 Fold 매칭 검증 및 행별 Fold 번호 매핑용 정보
# ------------------------------------------------------------

kf_groups = train['event_id']

assert not kf_groups.isna().any()

print('<시간사상 Group 확인>')
print(f'전체 학습 행 수 : {len(train):,}')
print(f'시간사상 개수   : {kf_groups.nunique():,}')
print()

# ------------------------------------------------------------
# 3-2. 기존 OOF에서 추출한 사상별 fold 배정 불러오기
# ------------------------------------------------------------

N_SPLITS = 5

kf_fold_map = pd.read_parquet(
    PROJECT_ROOT / 'data' / '0_kf_train_match.parquet'
)

# 검증
assert kf_fold_map.columns.tolist() == ['event_id', 'fold_num']
assert kf_fold_map['event_id'].is_unique
assert set(kf_fold_map['fold_num']) == set(range(1, N_SPLITS + 1))  # 1 ~ N_SPLITS 까지
assert not kf_fold_map.isna().any().any()  # 전체에 결측치 없어야 함
assert set(kf_groups) == set(kf_fold_map['event_id'])  # 실제 'event_id'와 매핑규칙이 동일해야함

# 시리즈.map() 안에는 기준컬럼이 인덱스이면서 치환하고자 하는 컬럼이 하나인 시리즈가 와야 함
# 시리즈.map()의 결과물은 시리즈의 값을 매핑규칙(시리즈의 컬럼이 인덱스로 된)에 따라 바꿔줌
kf_fold_num = kf_groups.map(  # kf_groups는 event_id 컬럼 전체, 여기에 매핑 시작
    kf_fold_map.set_index('event_id')['fold_num']
    )  # 결과물: kf_groups(train['event_id'])의 'event_id'가 'fold_num'으로 바뀜
assert kf_fold_num.notna().all()

import numpy as np
# 행 위치 배열만 보관한다. 데이터셋은 복사하지 않는다.
# 학습·진단·OOF fold 기록에서 모두 같은 분할을 재사용한다.
# np.flatnonzero(조건문): 조건을 만족하는 값의 숫자인덱스
kf_splits = [
    (
        np.flatnonzero(kf_fold_num.to_numpy() != fold),
        np.flatnonzero(kf_fold_num.to_numpy() == fold)
    )
    for fold in range(1, N_SPLITS + 1)
]

# ------------------------------------------------------------
# 3-3. Fold 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(
    kf_splits,
    start=1
):
    # 고정한 분할의 사상 중복 여부 검증
    train_groups = set(kf_groups.iloc[train_idx])  # 분할된 event_id
    valid_groups = set(kf_groups.iloc[valid_idx])  # 분할된 event_id

    overlap = train_groups & valid_groups  # 교집합은 0이어야함

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '
        f'valid events={len(valid_groups):,} | '
        f'event overlap={len(overlap)}'
    )

<시간사상 Group 확인>
전체 학습 행 수   : 616,293
시간사상 개수     : 98

Fold 1 | train=493,297 | valid=122,996 | train events=82 | valid events=16 | event overlap=0
Fold 2 | train=492,894 | valid=123,399 | train events=77 | valid events=21 | event overlap=0
Fold 3 | train=492,894 | valid=123,399 | train events=77 | valid events=21 | event overlap=0
Fold 4 | train=493,077 | valid=123,216 | train events=78 | valid events=20 | event overlap=0
Fold 5 | train=493,010 | valid=123,283 | train events=78 | valid events=20 | event overlap=0


In [ ]:
# ------------------------------------------------------------
# 3-4. K-Fold 교차검증
# ------------------------------------------------------------

# 각각 예측데이터 저장해서 OOF도 만들고 플롯도 해야함

# 메모리 사용을 줄이고, 효율적 코딩을 위해서는 폴드별로 데이터셋을 저장 X
# 데이터 분할 + 모델링을 한 세트로 for문을 구현

# 독립변수 정의
cols2 = origin_cols + wl_diff_cols + ['station_id']
X_train = train[cols2].copy()  # 'station_id' 문자열 변환 예정이므로 copy() 필요
X_val = val[cols2].copy()

X_train['station_id'] = X_train['station_id'].astype(str)
X_val['station_id'] = X_val['station_id'].astype(str)

# K-Fold 설계
kf_results = []  # Fold별 평가점수 등 담을 리스트
kf_oof_pred = np.full(len(X_train), np.nan)  # 빈 배열 만들기

for fold_num, (train_idx, val_idx) in enumerate(
    kf_splits,
    start=1
):
    # fold별 데이터 분할
    X_kf_tr = X_train.iloc[train_idx]
    X_kf_va = X_train.iloc[val_idx]

    y_kf_tr = y_train.iloc[train_idx]
    y_kf_va = y_train.iloc[val_idx]

    # 모델 정의
    model_cat_kf = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        cat_features=['station_id']  # CatBoost 전용 범주형변수 매개변수
    )

    # 학습
    model_cat_kf.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 예측
    pred_kf_va = model_cat_kf.predict(X_kf_va)
    pred_kf_tr = model_cat_kf.predict(X_kf_tr)
    kf_oof_pred[val_idx] = pred_kf_va

    # 평가
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    # 평가 결과 저장
    kf_results.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_cat_kf.best_iteration_
    })

    # 결과 출력
    print(
        f'Fold {fold_num} | '
        f'Val={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_cat_kf.best_iteration_}'
    )

# oof_RMSE 구하기
# 예측값 누락 확인
print(pd.isnull(kf_oof_pred).sum())

# 전체 OOF RMSE
oof_rmse = root_mean_squared_error(y_train, kf_oof_pred)

print(f'OOF RMSE: {oof_rmse:.4f}')

In [ ]:
# ====================================================================================
# 5. 모델15 (최종검증: CatBoost base model + iteration=3000 + early_stopping_rounds=100)
# ====================================================================================

# 모델 정의
model_base_iter_3000 = CatBoostRegressor(
    iterations=3000,
    random_state=42
)

# 모델 학습
model_base_iter_3000.fit(
    X_train_with_station_fe,
    y_train,
    early_stopping_rounds=100,
    eval_set=(X_val_with_station_fe, y_val),
    verbose=200
)

pred_base_iter_3000 = model_base_iter_3000.predict(X_val_with_station_fe)
pred_train_base_iter_3000 = model_base_iter_3000.predict(X_train_with_station_fe)

print('Best iteration:', model_base_iter_3000.best_iteration_)
print('Tree count:', model_base_iter_3000.tree_count_)
print('Best score:', model_base_iter_3000.best_score_['validation']['RMSE'])
print()

rmse_val_base_iter_3000 = root_mean_squared_error(
    y_val,
    pred_base_iter_3000
)

rmse_train_base_iter_3000 = root_mean_squared_error(
    y_train,
    pred_train_base_iter_3000
)

print('val RMSE: ', rmse_val_base_iter_3000)
print('train RMSE: ', rmse_train_base_iter_3000)
print(
    'Gap: ',
    rmse_val_base_iter_3000 - rmse_train_base_iter_3000
)


In [ ]:
# ============================================================
# 7. 모델15 (추가: OHE 적용 여부에 따른 RMSE)
# ============================================================
# CatBoost는 자체 인코딩이 가능함
# OHE 없이 한번 돌려보고 OHE 쓸지말지 결정 필요

# 모델은 model_base_iter_3000 그대로 사용

# station_id가 포함된 데이터셋 만들어야함, 데이터 수정예정이므로 .copy()로 기존데이터와의 연결 차단
X_train_with_station = train[num_cols_fe + ['station_id']].copy()
X_val_with_station = val[num_cols_fe + ['station_id']].copy()

# CatBoost 전용 범주형 변수 취급: str이나 int로 자료형 변환(.astype(str))
X_train_with_station['station_id'] = X_train_with_station['station_id'].astype(str)
X_val_with_station['station_id'] = X_val_with_station['station_id'].astype(str)

# 모델 정의
model_base_no_ohe = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']  # CatBoost 전용 범주형변수 매개변수
)

# 학습
model_base_no_ohe.fit(
    X_train_with_station,
    y_train,
    early_stopping_rounds=100,
    eval_set=(X_val_with_station, y_val),
    verbose=200
)

# 예측
pred_val_no_ohe = model_base_no_ohe.predict(X_val_with_station)
pred_train_no_ohe = model_base_no_ohe.predict(X_train_with_station)

print('Best iteration:', model_base_no_ohe.best_iteration_)
print('Tree count:', model_base_no_ohe.tree_count_)
print('Best score:', model_base_no_ohe.best_score_['validation']['RMSE'])
print()

rmse_val_no_ohe = root_mean_squared_error(
    y_val, pred_val_no_ohe,
)
rmse_train_no_ohe = root_mean_squared_error(
    y_train, pred_train_no_ohe
)

print('val RMSE: ', rmse_val_no_ohe)
print('train RMSE: ', rmse_train_no_ohe)
print(
    'Gap: ',
    rmse_val_no_ohe - rmse_train_no_ohe
)
